## Import packages


In [ ]:
%pip install -q xgboost

In [ ]:
%pip install mlflow sagemaker-mlflow

In [ ]:
import pandas as pd
import numpy as np 
import json
import joblib
import xgboost as xgb
import sagemaker
import boto3
import os
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from time import gmtime, strftime, sleep
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

import warnings
warnings.filterwarnings('ignore')

In [ ]:
target_col = 'y'

In [ ]:
%store target_col

In [ ]:
session = sagemaker.Session()
sm = session.sagemaker_client


## Load Data

In [ ]:
file_source = "local"
file_name = "bank-additional-full.csv"
input_path = "./data/bank-additional" 
output_path = "./data"

In [ ]:
if file_source != "local":
    session.download_data(
        path=os.path.join(input_path, ""), 
        bucket=bucket_name,
        key_prefix=f"{bucket_prefix}/input/{file_name}"
    )

## Tracking experiments with SageMaker and MLflow integration

In [ ]:
if file_source == "local":
    # Check that the MLflow server is in the status 'Created'
    sm = boto3.client("sagemaker")
    
    while sm.describe_mlflow_tracking_server(TrackingServerName=mlflow_name)['TrackingServerStatus'] != 'Created':
        print(f"The MLflow server {mlflow_name} is not in the status 'Created'")
        sleep(30)
    else:
        print(f"Using server {mlflow_name}")

In [ ]:
mlflow.set_tracking_uri(mlflow_arn)

In [ ]:
experiment_suffix = strftime('%d-%H-%M-%S', gmtime())
experiment_name = f"from-idea-to-prod-experiment-{experiment_suffix}"
registered_model_name = f"from-idea-to-prod-experiment-model-{experiment_suffix}"

In [ ]:
%store experiment_name

In [ ]:
experiment = mlflow.set_experiment(experiment_name=experiment_name)

## Feature Engineering

In [ ]:
# Indicator variable to capture when pdays takes a value of 999
df_data["no_previous_contact"] = np.where(df_data["pdays"] == 999, 1, 0)

# Indicator for individuals not actively employed
df_data["not_working"] = np.where(
    np.in1d(df_data["job"], ["student", "retired", "unemployed"]), 1, 0
)

# remove unnecessary data
df_model_data = df_data.drop(
    ["duration", "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"],
    axis=1,
)


bins = [18, 30, 40, 50, 60, 70, 90]
labels = ['18-29', '30-39', '40-49', '50-59', '60-69', '70-plus']

df_model_data['age_range'] = pd.cut(df_model_data.age, bins, labels=labels, include_lowest=True)
df_model_data = pd.concat([df_model_data, pd.get_dummies(df_model_data['age_range'], prefix='age', dtype=int)], axis=1)
df_model_data.drop('age', axis=1, inplace=True)
df_model_data.drop('age_range', axis=1, inplace=True)

scaled_features = ['pdays', 'previous', 'campaign']
df_model_data[scaled_features] = MinMaxScaler().fit_transform(df_model_data[scaled_features])

df_model_data = pd.get_dummies(df_model_data, dtype=int)  # Convert categorical variables to sets of indicators

# Replace "y_no" and "y_yes" with a single label column, and bring it to the front:
df_model_data = pd.concat(
    [
        df_model_data["y_yes"].rename(target_col),
        df_model_data.drop(["y_no", "y_yes"], axis=1),
    ],
    axis=1,
)

## Split data

In [ ]:
# Shuffle and splitting dataset
train_data, validation_data, test_data = np.split(
    df_model_data.sample(frac=1, random_state=1729),
    [int(0.7 * len(df_model_data)), int(0.9 * len(df_model_data))],
)

print(f"Data split > train:{train_data.shape} | validation:{validation_data.shape} | test:{test_data.shape}")

In [ ]:
# Save data to Studio filesystem
train_data.to_csv(os.path.join(output_path, "train.csv"), index=False, header=False)
validation_data.to_csv(os.path.join(output_path, "validation.csv"), index=False, header=False)
test_data.to_csv(os.path.join(output_path, "test.csv"), index=False, header=False)

## Model training and validation

In [ ]:
train_features = train_data.drop(target_col, axis=1)
train_label = pd.DataFrame(train_data[target_col])

dtrain = xgb.DMatrix(train_features, label=train_label)

In [ ]:
hyperparams = {
                "max_depth": 5,
                "eta": 0.5,
                "alpha": 2.5,
                "objective": "binary:logistic",
                "subsample" : 0.8,
                "colsample_bytree" : 0.8,
                "min_child_weight" : 3
              }

num_boost_round = 150
nfold = 3
early_stopping_rounds = 10

In [ ]:
# Cross-validate on training data
cv_results = xgb.cv(
    params=hyperparams,
    dtrain=dtrain,
    num_boost_round=num_boost_round,
    nfold=nfold,
    early_stopping_rounds=early_stopping_rounds,
    metrics=["auc"],
    seed=10,
)

In [ ]:
metrics_data = {
    "binary_classification_metrics": {
        "validation:auc": {
            "value": cv_results.iloc[-1]["test-auc-mean"],
            "standard_deviation": cv_results.iloc[-1]["test-auc-std"]
        },
        "train:auc": {
            "value": cv_results.iloc[-1]["train-auc-mean"],
            "standard_deviation": cv_results.iloc[-1]["train-auc-std"]
        },
    }
}

In [ ]:
print(f"Cross-validated train-auc:{cv_results.iloc[-1]['train-auc-mean']:.2f}")
print(f"Cross-validated validation-auc:{cv_results.iloc[-1]['test-auc-mean']:.2f}")

In [ ]:
test_features = test_data.drop(target_col, axis=1)
test_label = pd.DataFrame(test_data[target_col])
dtest = xgb.DMatrix(test_features, label=test_label)

## Create an experiment run

Create a new run using the `mlflow.start_run()` API and call the `log_params()` and `log_artifact()` MLflow logging functions to record information into the run. Note that `mlflow.log_artifact()` uploads a local file to the MLflow artifact store under the S3 URI that you specified when you created the MLflow server
<br>
You can also use `log_input()` method to persistently log a dataset to the MLflow artifact store

run_suffix = strftime('%d-%H-%M-%S', gmtime())

with mlflow.start_run(
    run_name=f"feature-engineering-{run_suffix}",
    description="feature-engineering in the notebook 01 ideation") as run:
    mlflow.log_params(
        {
            "train": 0.7,
            "validate": 0.2,
            "test": 0.1
        }
    )
    # Log input dataset metadata and output
    mlflow.log_artifact(local_path=os.path.join(input_path, file_name))
    mlflow.log_artifact(local_path=os.path.join(output_path, "train.csv"))
    mlflow.log_artifact(local_path=os.path.join(output_path, "validation.csv"))
    mlflow.log_artifact(local_path=os.path.join(output_path, "test.csv"))

## Train a model

Use MLflow model flavour and logging functions to log parameters, model, model metrics, and various metadata in your experiment runs

In [ ]:
run_suffix = strftime('%d-%H-%M-%S', gmtime())
max_metric = 0.0
best_model_run_id = 0

with mlflow.start_run(
    run_name=f"training-{run_suffix}",
    description=f"Fit estimator with different max_depth") as parent_run:
    mlflow.set_tags({'mlflow.user':user_profile_name})
    
    # Train the model for different max_depth values
    for i, d in enumerate([2, 5, 10, 15, 20]):
        hyperparams["max_depth"] = d
        print(f"Fit estimator with max_depth={d}")
    
        with mlflow.start_run(
            run_name=f"max_depth={d}",
            description=f"Fit estimator with max_depth={d}",
            nested=True) as child_run:
            mlflow.set_tags({'mlflow.user':user_profile_name})
            
            mlflow.xgboost.autolog(log_model_signatures=False, log_datasets=False)
            
            # Train the model
            model = xgb.train(
                params=hyperparams, 
                dtrain=dtrain, 
                evals = [(dtrain,'train'), (dtest,'eval')], 
                num_boost_round=num_boost_round, 
                early_stopping_rounds=early_stopping_rounds, 
                verbose_eval = 0
            )
    
            # Calculate metrics
            test_auc = roc_auc_score(test_label, model.predict(dtest))
            train_auc = roc_auc_score(train_label, model.predict(dtrain))
            
            # Log hyperparameters and metrics to the run
            mlflow.log_params(hyperparams)
            mlflow.log_metrics({"test_auc":test_auc, "train_auc":train_auc}, step=i)
    
            if test_auc > max_metric:
                best_model_run_id = child_run.info.run_id
                max_metric = test_auc
    
            print(f"Test AUC: {test_auc:.4f} | Train AUC: {train_auc:.4f}")

## Register the mode in the MLflow model registry

Use MLflow model registry to register a model. Here, we use `mlflow.register_model()` API to register the best model as a new version under the registered model name. `mlflow.register_model()` also automatically registers a model with the SageMaker Model Registry. When registering the MLflow model, a corresponding model package group and model package version are created in SageMaker. 

In [ ]:
model_uri = f"runs:/{best_model_run_id}/xgboost"
mv = mlflow.register_model(model_uri, registered_model_name)